# Generative Try-On System Experiments


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Project root could not be resolved from the notebook location.')


PROJECT_ROOT = find_project_root(Path.cwd())
GENERATIVE_SYSTEM_ROOT = PROJECT_ROOT / 'backend' / 'systems' / 'generative_tryon'
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'

if str(GENERATIVE_SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERATIVE_SYSTEM_ROOT))

PROJECT_ROOT

In [ ]:
from generative_app.bootstrap_static import ensure_static_2d_on_path

ensure_static_2d_on_path()

from generative_app.core.generative_tryon_pipeline import prepare_generative_tryon_package
from generative_app.config import OUTPUT_ROOT

from support_app.core.asset_bank import asset_bank_summary, get_asset_by_id
from support_app.core.face_analyzer import analyze_face_image
from support_app.core.hair_segmentation import predict_hair_mask_image, save_predicted_hair_mask


def show_image(image_path: str | Path, title: str | None = None, figsize=(5, 5)):
    image_path = Path(image_path)
    with Image.open(image_path) as image:
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(image)
        ax.axis('off')
        if title:
            ax.set_title(title)
        plt.show()


print('Project root:', PROJECT_ROOT)
print('Generative output root:', OUTPUT_ROOT)
print('Notebook Python:', sys.executable)
print('Venv Python:', VENV_PYTHON)

In [ ]:
INPUT_IMAGE_PATH = PROJECT_ROOT / 'backend' / 'outputs' / 'uploads' / 'image2.jpg'
ASSET_IDS = [
    'celeba_full_hair_000082',
    'celeba_full_hair_000111',
]

INPUT_IMAGE_PATH, ASSET_IDS

In [ ]:
bank = asset_bank_summary()
print('Active asset bank:', bank['metadata_dir'])
print('Asset count:', bank['asset_count'])

asset_rows = []
for asset_id in ASSET_IDS:
    asset = get_asset_by_id(asset_id)
    if asset is None:
        raise ValueError(f'Asset not found: {asset_id}')
    attrs = asset.normalized_attributes
    asset_rows.append({
        'asset_id': asset.asset_id,
        'length': attrs.length,
        'curl': attrs.curl,
        'bang': attrs.bang,
        'volume': attrs.volume,
        'side_hair': attrs.side_hair,
        'color': attrs.color,
        'style_family': attrs.style_family,
    })

display(pd.DataFrame(asset_rows))

In [ ]:
if not INPUT_IMAGE_PATH.exists():
    raise FileNotFoundError(f'Input image not found: {INPUT_IMAGE_PATH}')

analysis = analyze_face_image(INPUT_IMAGE_PATH)
print('Face detected:', analysis.face_detected)
print('Face bbox:', analysis.face_bbox)

show_image(INPUT_IMAGE_PATH, title='Subject Input', figsize=(4, 5))

In [ ]:
existing_mask_path = PROJECT_ROOT / 'backend' / 'outputs' / 'predictions' / 'hair_masks' / f'{INPUT_IMAGE_PATH.stem}_hair_mask.png'

try:
    with Image.open(INPUT_IMAGE_PATH) as source_image:
        subject_hair_mask = predict_hair_mask_image(source_image)
    mask_path = save_predicted_hair_mask(subject_hair_mask, INPUT_IMAGE_PATH.stem)
    print('Saved fresh mask:', mask_path)
except ModuleNotFoundError as exc:
    if not existing_mask_path.exists():
        raise RuntimeError(
            'Live hair-mask prediction requires PyTorch, and no saved fallback mask was found. '
            'Install the project root requirements.txt dependencies or provide a saved mask.'
        ) from exc
    print(f'Using existing saved mask because live prediction is unavailable: {existing_mask_path}')
    mask_path = existing_mask_path
    subject_hair_mask = Image.open(mask_path).convert('L')

show_image(mask_path, title='Subject Hair Mask', figsize=(4, 5))

In [ ]:
package_runs = []

for asset_id in ASSET_IDS:
    asset = get_asset_by_id(asset_id)
    package = prepare_generative_tryon_package(
        INPUT_IMAGE_PATH,
        analysis,
        asset,
        subject_hair_mask=subject_hair_mask,
    )
    if package is None:
        raise RuntimeError(f'Failed to prepare package for {asset_id}')

    manifest = json.loads(Path(package['manifest_path']).read_text(encoding='utf-8'))
    prompt_text = Path(package['prompt_path']).read_text(encoding='utf-8')
    attrs = asset.normalized_attributes

    package_runs.append({
        'asset_id': asset.asset_id,
        'length': attrs.length,
        'curl': attrs.curl,
        'style_family': attrs.style_family,
        'preview_path': package['preview_path'],
        'erased_subject_path': package['erased_subject_path'],
        'inpaint_mask_path': package['inpaint_mask_path'],
        'reference_image_path': package['reference_image_path'],
        'reference_mask_path': package['reference_mask_path'],
        'manifest_path': package['manifest_path'],
        'prompt_path': package['prompt_path'],
        'prompt_text': prompt_text,
        'package_dir': str(Path(package['preview_path']).parent),
        'face_bbox': manifest['subject']['anchors']['face_bbox'],
    })

package_df = pd.DataFrame(package_runs)
display(package_df[['asset_id', 'length', 'curl', 'style_family', 'package_dir']])

In [ ]:
for row in package_runs:
    print(row['asset_id'])
    show_image(row['preview_path'], title=f"{row['asset_id']} | Prototype Preview", figsize=(8, 6))

In [ ]:
selected = package_runs[0]
manifest = json.loads(Path(selected['manifest_path']).read_text(encoding='utf-8'))

print('Selected asset:', selected['asset_id'])
print('\nGeneration prompt:\n')
print(selected['prompt_text'])

manifest

In [ ]:
show_image(selected['erased_subject_path'], title='Erased Subject', figsize=(4, 5))
show_image(selected['inpaint_mask_path'], title='Inpaint Mask', figsize=(4, 5))
show_image(selected['reference_image_path'], title='Reference Hairstyle', figsize=(4, 5))
show_image(selected['reference_mask_path'], title='Reference Mask', figsize=(4, 5))

## Final Generative Edit

This section runs the heavy diffusion step through the project `.venv` in a subprocess. That keeps the notebook usable even if the active kernel is not the venv kernel.

Default model:
- `runwayml/stable-diffusion-inpainting`

Notes:
- the first run downloads model weights unless you switch to a local model path
- CPU execution is possible but will be much slower than CUDA


In [ ]:
DEFAULT_INPAINT_MODEL_ID = 'runwayml/stable-diffusion-inpainting'
FINAL_STEPS = 30
FINAL_GUIDANCE_SCALE = 7.5
FINAL_STRENGTH = 0.99
FINAL_MAX_SIDE = 768
FINAL_EXTRA_PROMPT = 'Keep the subject realistic and retain a natural forehead transition.'


def _decode_output(payload: bytes | None) -> str:
    if payload is None:
        return ''
    return payload.decode('utf-8', errors='replace')


def run_final_generation(
    manifest_path: str | Path,
    model_id: str = DEFAULT_INPAINT_MODEL_ID,
    steps: int = FINAL_STEPS,
    guidance_scale: float = FINAL_GUIDANCE_SCALE,
    strength: float = FINAL_STRENGTH,
    max_side: int = FINAL_MAX_SIDE,
    extra_prompt: str | None = FINAL_EXTRA_PROMPT,
):
    manifest_path = Path(manifest_path)
    if not VENV_PYTHON.exists():
        raise FileNotFoundError(f'Venv Python not found: {VENV_PYTHON}')

    command = [
        str(VENV_PYTHON),
        '-m',
        'generative_app.core.generative_inpaint_runner',
        '--manifest',
        str(manifest_path),
        '--model-id',
        model_id,
        '--steps',
        str(steps),
        '--guidance-scale',
        str(guidance_scale),
        '--strength',
        str(strength),
        '--max-side',
        str(max_side),
    ]
    if extra_prompt:
        command.extend(['--extra-prompt', extra_prompt])

    completed = subprocess.run(
        command,
        cwd=GENERATIVE_SYSTEM_ROOT,
        capture_output=True,
        text=False,
        check=False,
    )
    stdout_text = _decode_output(completed.stdout)
    stderr_text = _decode_output(completed.stderr)

    if stderr_text.strip():
        print(stderr_text)

    if completed.returncode != 0:
        raise RuntimeError(
            f'Final generation failed with exit code {completed.returncode}.\n\n'
            f'STDERR:\n{stderr_text}\n\nSTDOUT:\n{stdout_text}'
        )

    return json.loads(stdout_text)


selected['manifest_path']

In [ ]:
final_result = run_final_generation(selected['manifest_path'])
final_result

In [ ]:
show_image(final_result['output_image_path'], title='Final Generated Try-On', figsize=(4, 5))
json.loads(Path(final_result['metadata_path']).read_text(encoding='utf-8'))